In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Project Configuration

This section defines the project root and all persistent storage locations.

All paths are derived from a single project root to avoid hard-coded paths
throughout the project.

In [ ]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path("/content/drive/MyDrive/ForecastOpti")

# Data
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Figures
FIGURES_DIR = PROJECT_ROOT / "figures"
EDA_FIGURES_DIR = FIGURES_DIR / "eda"
FORECASTING_FIGURES_DIR = FIGURES_DIR / "forecasting"
OPTIMIZATION_FIGURES_DIR = FIGURES_DIR / "optimization"

# Models
MODELS_DIR = PROJECT_ROOT / "models"

# Notebooks
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

# Outputs
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FORECAST_OUTPUT_DIR = OUTPUTS_DIR / "forecasts"
OPTIMIZATION_OUTPUT_DIR = OUTPUTS_DIR / "optimization"
EVALUATION_OUTPUT_DIR = OUTPUTS_DIR / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

Project root: /content/drive/MyDrive/ForecastOpti


## 3. Directory Initialization

Create all required project directories.

The operation is idempotent: running this cell multiple times
will not overwrite or delete existing files.

In [ ]:
PROJECT_DIRECTORIES = [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    EDA_FIGURES_DIR,
    FORECASTING_FIGURES_DIR,
    OPTIMIZATION_FIGURES_DIR,
    MODELS_DIR,
    NOTEBOOKS_DIR,
    OUTPUTS_DIR,
    FORECAST_OUTPUT_DIR,
    OPTIMIZATION_OUTPUT_DIR,
    EVALUATION_OUTPUT_DIR,
]

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✓ Verified {len(PROJECT_DIRECTORIES)} project directories.")

✓ Verified 13 project directories.


## 4. Project Structure Validation

Verify that the expected project directories exist before
continuing with the ML pipeline.

In [ ]:
missing_directories = [
    directory
    for directory in PROJECT_DIRECTORIES
    if not directory.exists()
]

if missing_directories:
    print("Missing directories:")
    for directory in missing_directories:
        print(f"  - {directory}")
else:
    print("✓ All project directories are available.")

✓ All project directories are available.


## 5. Dependencies

Install the core libraries required for data analysis,
statistical forecasting, machine learning, optimization,
and data storage.

Deep learning dependencies will use the PyTorch environment
provided by Google Colab.

In [ ]:
%pip install -q \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    statsmodels \
    xgboost \
    lightgbm \
    catboost \
    pyarrow

## 6. Library Imports

Import the core libraries used throughout the project setup.

In [ ]:
import os
import platform
import random
import sys

import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels
import torch
import xgboost
import lightgbm
import catboost

## 7. Reproducibility

Configure deterministic random seeds where supported.

A fixed seed allows experiments to be reproduced more consistently.

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"✓ Random seed configured: {SEED}")

✓ Random seed configured: 42


## 8. Raw Dataset Validation

Verify that the primary dataset exists in the expected location.

Detailed data exploration will be performed in `02_data_understanding.ipynb`.

In [ ]:
DATASET_PATH = RAW_DATA_DIR / "retail_sales.csv"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATASET_PATH}"
    )

file_size_mb = DATASET_PATH.stat().st_size / (1024 ** 2)

print("✓ Dataset found")
print(f"Path: {DATASET_PATH}")
print(f"Size: {file_size_mb:.2f} MB")

✓ Dataset found
Path: /content/drive/MyDrive/ForecastOpti/data/raw/retail_sales.csv
Size: 186.11 MB


In [ ]:
DATA_PREVIEW_ROWS = 10_000

preview_df = pd.read_csv(
    DATASET_PATH,
    nrows=DATA_PREVIEW_ROWS
)

print(f"Preview rows: {len(preview_df):,}")
print(f"Columns: {len(preview_df.columns)}")

preview_df.head()

Preview rows: 10,000
Columns: 8


,date,store_id,item_id,sales,price,promo,weekday,month
0,2019-01-01,store_1,item_1,41,21.30,0,1,1
1,2019-01-02,store_1,item_1,53,21.30,0,2,1
2,2019-01-03,store_1,item_1,39,21.30,0,3,1
3,2019-01-04,store_1,item_1,35,21.30,0,4,1
4,2019-01-05,store_1,item_1,51,17.04,1,5,1


In [ ]:
preview_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      10000 non-null  object 
 1   store_id  10000 non-null  object 
 2   item_id   10000 non-null  object 
 3   sales     10000 non-null  int64  
 4   price     10000 non-null  float64
 5   promo     10000 non-null  int64  
 6   weekday   10000 non-null  int64  
 7   month     10000 non-null  int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 625.1+ KB


In [ ]:
preview_df.columns.tolist()

['date', 'store_id', 'item_id', 'sales', 'price', 'promo', 'weekday', 'month']

In [ ]:
setup_summary = {
    "project_root": str(PROJECT_ROOT),
    "dataset": str(DATASET_PATH),
    "dataset_size_mb": round(file_size_mb, 2),
    "seed": SEED,
}

pd.Series(setup_summary)

,0
project_root,/content/drive/MyDrive/ForecastOpti
dataset,/content/drive/MyDrive/ForecastOpti/data/raw/r...
dataset_size_mb,186.11
seed,42


## 9. Final Setup Validation

Validate the critical components required to start the ForecastOpti
data analysis pipeline.

In [ ]:
EXPECTED_COLUMNS = {
    'date',
    'store_id',
    'item_id',
    'sales',
    'price',
    'promo',
    'weekday',
    'month',
}

validation_checks = {
    "project_root_exists": PROJECT_ROOT.exists(),
    "dataset_exists": DATASET_PATH.exists(),
    "raw_data_directory_exists": RAW_DATA_DIR.exists(),
    "processed_data_directory_exists": PROCESSED_DATA_DIR.exists(),
    "models_directory_exists": MODELS_DIR.exists(),
    "outputs_directory_exists": OUTPUTS_DIR.exists(),
    "required_columns_present": EXPECTED_COLUMNS.issubset(
        set(preview_df.columns)
    ),
}

failed_checks = [
    check
    for check, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    raise RuntimeError(
        "Setup validation failed:\n"
        + "\n".join(f"- {check}" for check in failed_checks)
    )

print("✓ ForecastOpti setup validation passed.")
print("✓ Environment is ready for data analysis.")

✓ ForecastOpti setup validation passed.
✓ Environment is ready for data analysis.


In [ ]:
config_content = '''
from pathlib import Path

# Project
PROJECT_ROOT = Path("/content/drive/MyDrive/ForecastOpti")


# Data
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


# Figures
FIGURES_DIR = PROJECT_ROOT / "figures"
EDA_FIGURES_DIR = FIGURES_DIR / "eda"
FORECASTING_FIGURES_DIR = FIGURES_DIR / "forecasting"
OPTIMIZATION_FIGURES_DIR = FIGURES_DIR / "optimization"


# Models
MODELS_DIR = PROJECT_ROOT / "models"


# Notebooks
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"


# Outputs
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FORECAST_OUTPUT_DIR = OUTPUTS_DIR / "forecasts"
OPTIMIZATION_OUTPUT_DIR = OUTPUTS_DIR / "optimization"
EVALUATION_OUTPUT_DIR = OUTPUTS_DIR / "evaluation"


# Dataset
DATASET_PATH = RAW_DATA_DIR / "retail_sales.csv"


# Reproducibility
SEED = 42
'''

SRC_DIR = PROJECT_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)

(CONFIG_PATH := SRC_DIR / "config.py").write_text(
    config_content.strip() + "\n"
)

print(f"✓ Created: {CONFIG_PATH}")

✓ Created: /content/drive/MyDrive/ForecastOpti/src/config.py


In [ ]:
INIT_PATH = SRC_DIR / "__init__.py"

INIT_PATH.write_text(
    '"""ForecastOpti source package."""\n'
)

print(f"✓ Created: {INIT_PATH}")

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    PROJECT_ROOT,
    DATASET_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
    SEED,
)

print("✓ ForecastOpti configuration imported successfully.")
print(f"Project : {PROJECT_ROOT}")
print(f"Dataset : {DATASET_PATH}")
print(f"Seed    : {SEED}")

✓ ForecastOpti configuration imported successfully.
Project : /content/drive/MyDrive/ForecastOpti
Dataset : /content/drive/MyDrive/ForecastOpti/data/raw/retail_sales.csv
Seed    : 42
